In [ ]:
#@title **Seq2Bio v1.0.0: Integrated Protein Sequence Analysis Pipeline** { display-mode: "form" }

#@markdown ### **From sequence to biological context – BLAST, taxonomy, images, maps, publications, phylogeny, and structure prediction in one notebook.**

#@markdown <center><img src="https://raw.githubusercontent.com/Vidhusv/seq2bio/main/probr.png" width="600"></center>

#@markdown ---
#@markdown #### **Links**
#@markdown * GitHub: [https://github.com/Vidhusv/seq2bio](https://github.com/Vidhusv/seq2bio)
#@markdown * Zenodo DOI: [10.5281/zenodo.19015565](https://doi.org/10.5281/zenodo.19015565)
#@markdown * Colab Notebook: [Open in Colab](https://colab.research.google.com/drive/16XlTDvXRkk1n6FZpICSYdnraF32TejR1?usp=sharing)

#@markdown ---
#@markdown #### **How to Cite**
#@markdown If you use Seq2Bio in your research, please cite it as:
#@markdown
#@markdown > Vijay, V. S. (2026). Seq2Bio: A Colab‑based tool for protein‑centric biological context retrieval (v2.0.0). Zenodo. [https://doi.org/10.5281/zenodo.19015565](https://doi.org/10.5281/zenodo.19015565)

#@markdown ---
print("Seq2Bio v2.0.0 loaded – your heroic octopus‑powered protein explorer is ready!")

In [ ]:
#@title Install system dependencies
!apt-get install -y clustalo
!pip install -q --upgrade pip

In [ ]:
#@title Install Python packages and imports
!pip install -q biopython==1.85 requests folium pandas matplotlib seaborn py3Dmol tqdm ipywidgets
!pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
!pip install -q --no-deps jax==0.4.23 jaxlib==0.4.23

# Import libraries
import os, time, subprocess, tempfile, json, pickle, glob, re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests
import folium
from folium.plugins import HeatMap
import py3Dmol
from tqdm import tqdm
from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.PDB import alphafold_db, MMCIFParser, PDBParser, is_aa, CaPPBuilder
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from google.colab import files, userdata
from IPython.display import HTML, display

print("All imports ready.")

In [35]:
#@title Configuration { run: "auto" }

# ----- Basic settings -----
Entrez_email = "" #@param {type:"string"}
max_hits = 10 #@param {type:"slider", min:5, max:20}
query_sequence = "" #@param {type:"string"}
pubmed_keywords = "toxin OR venom" #@param {type:"string"}
use_email_notification = True #@param {type:"boolean"}

# ----- Structure prediction settings (ColabFold) -----
model_type = "auto" #@param ["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3"]
num_recycles = "3" #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
recycle_early_stop_tolerance = "auto" #@param ["auto", "0.0", "0.5", "1.0"]
use_amber = True #@param {type:"boolean"}
use_templates = False #@param {type:"boolean"}
msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref", "single_sequence", "custom"]
pair_mode = "unpaired_paired" #@param ["unpaired_paired", "paired", "unpaired"]
save_all = False #@param {type:"boolean"}

# Store configuration in global variables
import sys
sys.modules[__name__].__dict__.update(locals())
print("Configuration saved.")

Configuration saved.


In [ ]:
#@title Helper Functions

# ---------- BLAST ----------
def run_blast(sequence, max_hits):
    """Run BLASTp and return parsed results."""
    print("Running BLAST...")
    result_handle = NCBIWWW.qblast("blastp", "nr", sequence, hitlist_size=max_hits*2)
    blast_records = NCBIXML.parse(result_handle)
    hits = []
    for record in blast_records:
        for alignment in record.alignments:
            for hsp in alignment.hsps:
                identity = hsp.identities / hsp.align_length
                if identity >= 0.7:
                    title = alignment.title
                    if '[' in title and ']' in title:
                        species = title.split('[')[-1].split(']')[0]
                    else:
                        species = title.split(',')[0].strip()
                    hits.append({
                        'accession': alignment.accession,
                        'title': title,
                        'species': species,
                        'identity': identity,
                        'length': hsp.align_length,
                        'query_seq': hsp.query,
                        'hit_seq': hsp.sbjct
                    })
    hits.sort(key=lambda x: x['identity'], reverse=True)
    return hits[:max_hits]

# ---------- NCBI Taxonomy ----------
def get_taxonomy(species_name):
    try:
        handle = Entrez.esearch(db="taxonomy", term=species_name)
        record = Entrez.read(handle)
        handle.close()
        if record["IdList"]:
            tax_id = record["IdList"][0]
            handle = Entrez.efetch(db="taxonomy", id=tax_id)
            tax_record = Entrez.read(handle)
            handle.close()
            lineage = tax_record[0]["Lineage"].split("; ")
            return {
                "kingdom": lineage[0] if len(lineage) > 0 else "",
                "phylum": lineage[1] if len(lineage) > 1 else "",
                "class": lineage[2] if len(lineage) > 2 else "",
                "order": lineage[3] if len(lineage) > 3 else "",
                "family": lineage[4] if len(lineage) > 4 else "",
                "genus": lineage[5] if len(lineage) > 5 else "",
                "species": tax_record[0]["ScientificName"]
            }
    except Exception as e:
        print(f"Taxonomy error for {species_name}: {e}")
    return {}

# ---------- iNaturalist Images ----------
def get_inaturalist_images(species):
    url = "https://api.inaturalist.org/v1/observations"
    params = {
        "taxon_name": species,
        "quality_grade": "research",
        "photos": True,
        "per_page": 3
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        images = []
        for obs in data.get('results', []):
            for photo in obs.get('photos', []):
                img_url = photo.get('url', '').replace('square', 'medium')
                if img_url:
                    images.append({
                        'url': img_url,
                        'attribution': photo.get('attribution', '')
                    })
        return images
    except Exception as e:
        print(f"iNaturalist error for {species}: {e}")
        return []

# ---------- iNaturalist Common Name ----------
def get_inat_common_name(species):
    url = "https://api.inaturalist.org/v1/taxa"
    params = {"q": species}
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        if data['total_results'] > 0:
            return data['results'][0].get('preferred_common_name', 'N/A')
    except Exception as e:
        print(f"iNaturalist taxon error for {species}: {e}")
    return 'N/A'

# ---------- GBIF Occurrences ----------
def get_gbif_occurrences(species):
    url = "https://api.gbif.org/v1/occurrence/search"
    params = {
        "scientificName": species,
        "hasCoordinate": True,
        "limit": 50
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        occurrences = []
        for occ in data.get('results', []):
            lat = occ.get('decimalLatitude')
            lon = occ.get('decimalLongitude')
            if lat is not None and lon is not None:
                loc_parts = []
                if occ.get('country'): loc_parts.append(occ['country'])
                if occ.get('stateProvince'): loc_parts.append(occ['stateProvince'])
                if occ.get('locality'): loc_parts.append(occ['locality'])
                location_name = ', '.join(loc_parts) if loc_parts else 'Unknown location'
                occurrences.append({
                    'lat': lat,
                    'lon': lon,
                    'location': location_name
                })
        return occurrences
    except Exception as e:
        print(f"GBIF error for {species}: {e}")
        return []

# ---------- PubMed ----------
def get_pubmed_articles(species, keywords):
    query = f"{species}[Organism] AND ({keywords})"
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=5)
        record = Entrez.read(handle)
        handle.close()
        if not record["IdList"]:
            return []
        handle = Entrez.esummary(db="pubmed", id=",".join(record["IdList"]))
        summaries = Entrez.read(handle)
        handle.close()
        articles = []
        for doc in summaries:
            articles.append({
                'pmid': doc['Id'],
                'title': doc.get('Title', 'No title'),
                'authors': ', '.join(doc.get('AuthorList', [])),
                'journal': doc.get('FullJournalName', ''),
                'year': doc.get('PubDate', '').split()[0] if doc.get('PubDate') else ''
            })
        return articles
    except Exception as e:
        print(f"PubMed error: {e}")
        return []

# ---------- Clustal Omega ----------
def run_clustalo(seq_dict):
    import os, tempfile, subprocess
    with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as f_in:
        for seq_id, seq in seq_dict.items():
            f_in.write(f">{seq_id}\n{seq}\n")
        infile = f_in.name
    outfile = infile + ".aln"
    treefile = infile + ".dnd"
    try:
        cmd = ['clustalo', '-i', infile, '-o', outfile, '--guidetree-out', treefile, '--force']
        result = subprocess.run(cmd, check=False, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Clustal Omega error: {result.stderr}")
            alignment = tree = ""
        else:
            alignment = open(outfile).read() if os.path.exists(outfile) else ""
            tree = open(treefile).read() if os.path.exists(treefile) else ""
    except Exception as e:
        print(f"Exception: {e}")
        alignment = tree = ""
    finally:
        for f in [infile, outfile, treefile]:
            if os.path.exists(f):
                os.unlink(f)
    return alignment, tree

# ---------- GBIF Images ----------
def get_gbif_images(species, max_images=3):
    try:
        taxon_url = f"https://api.gbif.org/v1/species/match?name={species}"
        taxon_response = requests.get(taxon_url, timeout=10)
        if taxon_response.status_code != 200:
            return []
        taxon_data = taxon_response.json()
        taxon_key = taxon_data.get('usageKey')
        if not taxon_key:
            return []
        images_url = f"https://api.gbif.org/v1/occurrence/search?media_type=StillImage&taxon_key={taxon_key}&limit={max_images}"
        img_response = requests.get(images_url, timeout=10)
        if img_response.status_code != 200:
            return []
        data = img_response.json()
        images = []
        for result in data.get('results', []):
            for media in result.get('media', []):
                if media.get('type') == 'StillImage':
                    images.append({
                        'url': media.get('identifier'),
                        'attribution': media.get('rightsHolder', media.get('publisher', 'GBIF'))
                    })
        return images
    except Exception as e:
        print(f"GBIF image error for {species}: {e}")
        return []

# ---------- GBIF Common Name ----------
def get_gbif_common_name(species):
    try:
        match_url = f"https://api.gbif.org/v1/species/match?name={species}"
        resp = requests.get(match_url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            usage_key = data.get('usageKey')
            if usage_key:
                vern_url = f"https://api.gbif.org/v1/species/{usage_key}/vernacularNames"
                vern_resp = requests.get(vern_url, timeout=10)
                if vern_resp.status_code == 200:
                    vern_data = vern_resp.json()
                    for item in vern_data.get('results', []):
                        if item.get('language') == 'eng':
                            return item.get('vernacularName', 'N/A')
    except Exception as e:
        print(f"GBIF common name error for {species}: {e}")
    return 'N/A'

# ---------- UniProt mapping ----------
def get_uniprot_from_accession(accession):
    if re.match(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9]', accession):
        return accession
    if accession.startswith(('NP_', 'XP_', 'YP_')):
        from_db = 'RefSeq_Protein'
    else:
        from_db = 'EMBL-GenBank-DDBJ_CDS'
    url = "https://rest.uniprot.org/idmapping/run"
    data = {'from': from_db, 'to': 'UniProtKB', 'ids': accession}
    try:
        resp = requests.post(url, data=data, timeout=30)
        resp.raise_for_status()
        job_id = resp.json().get('jobId')
        if not job_id:
            return None
        time.sleep(2)
        result_url = f"https://rest.uniprot.org/idmapping/stream/{job_id}"
        result_resp = requests.get(result_url, timeout=30)
        result_resp.raise_for_status()
        results = result_resp.json().get('results', [])
        return results[0]['to'] if results else None
    except:
        return None

# ---------- AlphaFold DB ----------
def check_alphafold_db(uniprot_id, out_dir='./structures'):
    os.makedirs(out_dir, exist_ok=True)
    try:
        predictions = list(alphafold_db.get_predictions(uniprot_id))
        if not predictions:
            return None
        pred = predictions[0]
        af_id = pred['entryId']
        cif_path = alphafold_db.download_cif_for(pred, directory=out_dir)
        pae_url = f"https://alphafold.ebi.ac.uk/files/{af_id}-predicted_aligned_error_v4.json"
        pae_path = None
        if requests.head(pae_url).status_code == 200:
            pae_data = requests.get(pae_url).text
            pae_path = os.path.join(out_dir, f"{af_id}_pae.json")
            with open(pae_path, 'w') as f:
                f.write(pae_data)
        return {'cif_path': cif_path, 'pae_path': pae_path, 'alphafold_id': af_id}
    except:
        return None

# ---------- Display CIF ----------
def display_cif(cif_path, title=""):
    with open(cif_path) as f:
        cif_data = f.read()
    view = py3Dmol.view(width=800, height=400)
    view.addModel(cif_data, 'cif')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    if title:
        print(title)
    return view.show()

# ---------- pLDDT plot ----------
def plot_plddt(cif_path, uniprot_id):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure('protein', cif_path)
    plddt = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if 'CA' in residue:
                    plddt.append(residue['CA'].get_bfactor())
    plt.figure(figsize=(12,4))
    plt.plot(plddt, color='#2c7fb8')
    plt.axhline(90, color='green', ls='--', alpha=0.5, label='Very high')
    plt.axhline(70, color='orange', ls='--', alpha=0.5, label='High')
    plt.axhline(50, color='red', ls='--', alpha=0.5, label='Low')
    plt.ylim(0,100)
    plt.xlabel('Residue')
    plt.ylabel('pLDDT')
    plt.title(f'pLDDT – {uniprot_id}')
    plt.legend()
    plt.show()

# ---------- PAE plot ----------
def plot_pae(pae_path, uniprot_id):
    if not pae_path or not os.path.exists(pae_path):
        return
    with open(pae_path) as f:
        pae_data = json.load(f)
    pae_mat = np.array(pae_data[0]['predicted_aligned_error'])
    plt.figure(figsize=(7,6))
    plt.imshow(pae_mat, cmap='viridis_r', vmin=0, vmax=30)
    plt.colorbar(label='PAE (Å)')
    plt.title(f'PAE – {uniprot_id}')
    plt.xlabel('Residue')
    plt.ylabel('Residue')
    plt.show()

# ---------- Run ColabFold (advanced) ----------
def run_colabfold(sequence, name, out_dir='./cf_predictions',
                  model_type='alphafold2_ptm', num_recycles=3,
                  recycle_early_stop_tolerance=0.0, use_amber=True,
                  use_templates=False, msa_mode='mmseqs2_uniref_env',
                  pair_mode='unpaired_paired', save_all=False):
    import subprocess, glob
    os.makedirs(out_dir, exist_ok=True)
    fasta = os.path.join(out_dir, f"{name}.fasta")
    with open(fasta, 'w') as f:
        f.write(f">{name}\n{sequence}\n")

    cmd = ['colabfold_batch', fasta, out_dir]

    # Add optional arguments
    if use_amber:
        cmd.append('--amber')
    if use_templates:
        cmd.append('--templates')
    if num_recycles != 'auto':
        cmd.extend(['--num-recycle', str(num_recycles)])
    if recycle_early_stop_tolerance != 'auto':
        cmd.extend(['--recycle-early-stop-tolerance', str(recycle_early_stop_tolerance)])
    if model_type != 'auto':
        cmd.extend(['--model-type', model_type])
    if msa_mode != 'mmseqs2_uniref_env':
        cmd.extend(['--msa-mode', msa_mode])
    if pair_mode != 'unpaired_paired':
        cmd.extend(['--pair-mode', pair_mode])
    if save_all:
        cmd.append('--save-all')

    print(f"  Running ColabFold for {name} with: {' '.join(cmd)}")
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        pdb_files = glob.glob(os.path.join(out_dir, f"{name}_unrelaxed_rank_*.pdb"))
        return pdb_files[0] if pdb_files else None
    except subprocess.CalledProcessError as e:
        print(f"  ❌ ColabFold failed with code {e.returncode}")
        print(f"  Error: {e.stderr}")
        return None
    except Exception as e:
        print(f"  ❌ Unexpected error: {e}")
        return None

# ---------- pLDDT table ----------
def create_plddt_table(cif_path, uniprot_id):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure('protein', cif_path)
    data = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if 'CA' in residue:
                    res_num = residue.id[1]
                    res_name = residue.resname
                    plddt = residue['CA'].get_bfactor()
                    data.append([res_num, res_name, plddt])
    df = pd.DataFrame(data, columns=['Residue', 'AA', 'pLDDT'])
    return df

# ---------- Ramachandran plot ----------
def plot_ramachandran(structure_file, file_format='cif', title=""):
    if file_format.lower() == 'pdb':
        parser = PDBParser(QUIET=True)
    else:
        parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure('protein', structure_file)

    phi_psi_data = []
    for model in structure:
        for chain in model:
            ppb = CaPPBuilder()
            for poly in ppb.build_peptides(chain):
                phi_psi = poly.get_phi_psi_list()
                for i, (phi, psi) in enumerate(phi_psi):
                    if phi is not None and psi is not None:
                        res = poly[i]
                        if is_aa(res):
                            phi_psi_data.append((np.degrees(phi), np.degrees(psi), res.resname))

    if not phi_psi_data:
        print("No phi/psi angles could be extracted.")
        return

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.fill_betweenx([-30, 10], -100, -30, alpha=0.15, color='blue', label='α-helix region')
    ax.fill_between([-180, -30], 90, 180, alpha=0.15, color='green', label='β-sheet region')
    ax.fill_between([30, 90], 30, 90, alpha=0.15, color='orange', label='Left-handed helix')

    colors = {'ALA': 'blue', 'GLY': 'red', 'PRO': 'green', 'others': 'gray'}
    for phi, psi, resname in phi_psi_data:
        if resname == 'GLY':
            color, marker, size = colors['GLY'], 's', 30
        elif resname == 'PRO':
            color, marker, size = colors['PRO'], '^', 30
        else:
            color, marker, size = colors['others'], 'o', 20
        ax.scatter(phi, psi, c=color, marker=marker, s=size, alpha=0.7, edgecolors='black', linewidth=0.5)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Other residues', markerfacecolor='gray', markersize=8),
        Line2D([0], [0], marker='s', color='w', label='Glycine', markerfacecolor='red', markersize=8),
        Line2D([0], [0], marker='^', color='w', label='Proline', markerfacecolor='green', markersize=8),
        Line2D([0], [0], color='blue', alpha=0.15, linewidth=10, label='α-helix region'),
        Line2D([0], [0], color='green', alpha=0.15, linewidth=10, label='β-sheet region'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')

    ax.set_xlabel('Phi (φ) [degrees]', fontsize=12)
    ax.set_ylabel('Psi (ψ) [degrees]', fontsize=12)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.axhline(0, color='black', linewidth=0.5, alpha=0.3)
    ax.axvline(0, color='black', linewidth=0.5, alpha=0.3)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(np.arange(-180, 181, 45))
    ax.set_yticks(np.arange(-180, 181, 45))

    if title:
        ax.set_title(f'Ramachandran Plot - {title}', fontsize=14)
    else:
        ax.set_title('Ramachandran Plot', fontsize=14)
    plt.tight_layout()
    plt.show()

    total = len(phi_psi_data)
    favored = sum(1 for phi, psi, _ in phi_psi_data
                  if (-120 <= phi <= -40 and 80 <= psi <= 180) or
                     (-100 <= phi <= -30 and -70 <= psi <= -10))
    print(f"Total residues analyzed: {total}")
    print(f"Residues in favored regions: {favored} ({favored/total*100:.1f}%)")

print("✅ All helper functions loaded.")

In [ ]:
#@title Run Full Analysis
# This cell runs the entire pipeline using the configuration above.

import time
import pickle
from tqdm import tqdm

# Set email for Entrez
Entrez.email = Entrez_email

# 1. BLAST
cache_file = "blast_results.pkl"
if os.path.exists(cache_file):
    with open(cache_file, 'rb') as f:
        hits = pickle.load(f)
    print("✅ Loaded cached BLAST results.")
else:
    hits = run_blast(query_sequence, max_hits)
    if hits:
        with open(cache_file, 'wb') as f:
            pickle.dump(hits, f)
    else:
        hits = []

if not hits:
    print("No hits with >70% identity found. Stopping.")
    # Create empty list to avoid later errors
    hits = []
else:
    print(f"Found {len(hits)} hits.")
    df_hits = pd.DataFrame([
        {"Species": h["species"], "Accession": h["accession"], "Identity": f"{h['identity']:.1%}"}
        for h in hits
    ])
    display(df_hits)

# 2. GBIF images & common names (for map tooltips)
species_images = {}
species_common = {}
for hit in tqdm(hits, desc="Fetching GBIF data"):
    sp = hit['species']
    images = get_gbif_images(sp)
    species_images[sp] = images[0]['url'] if images else None
    species_common[sp] = get_gbif_common_name(sp) or 'N/A'
    time.sleep(0.5)

# 3. Taxonomy & iNaturalist table
taxonomy_dict = {}
table_rows = []
for hit in tqdm(hits, desc="Processing iNaturalist data"):
    sp = hit['species']
    if sp not in taxonomy_dict:
        taxonomy_dict[sp] = get_taxonomy(sp)
    common = get_inat_common_name(sp)
    imgs = get_inaturalist_images(sp)
    img_html = f'<img src="{imgs[0]["url"]}" width="150">' if imgs else "No image"
    table_rows.append({"Species": sp, "Common Name": common, "Image": img_html})
    time.sleep(0.5)

if table_rows:
    display(HTML(pd.DataFrame(table_rows).to_html(escape=False, index=False)))
if taxonomy_dict:
    display(pd.DataFrame(taxonomy_dict).T)

# 4. Map
occurrences_dict = {}
all_coords = []
for hit in tqdm(hits, desc="Fetching occurrences"):
    sp = hit['species']
    if sp not in occurrences_dict:
        occ_data = get_gbif_occurrences(sp)
        occurrences_dict[sp] = occ_data
        all_coords.extend([(o['lat'], o['lon']) for o in occ_data])
        time.sleep(0.5)

if all_coords:
    mean_lat = sum(c[0] for c in all_coords) / len(all_coords)
    mean_lon = sum(c[1] for c in all_coords) / len(all_coords)
    m = folium.Map(location=[mean_lat, mean_lon], zoom_start=3)
    folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attr='Esri', name='Satellite').add_to(m)

    for sp, occ_list in occurrences_dict.items():
        img = species_images.get(sp)
        common = species_common.get(sp, 'N/A')
        for occ in occ_list:
            tooltip = f'<img src="{img}" width="80"> <b>{sp}</b><br><i>{common}</i><br><small>{occ["location"]}</small>' if img else f'<b>{sp}</b><br>{occ["location"]}'
            folium.CircleMarker([occ['lat'], occ['lon']], radius=3, color='red', fill=True, tooltip=tooltip).add_to(m)

    HeatMap(all_coords).add_to(m)
    folium.LayerControl().add_to(m)
    display(m)
else:
    print("No coordinates found.")

# 5. Publications
publications_dict = {}
for hit in tqdm(hits, desc="Fetching publications"):
    sp = hit['species']
    if sp not in publications_dict:
        pubs = get_pubmed_articles(sp, pubmed_keywords)
        publications_dict[sp] = pubs
        time.sleep(0.5)

for sp, pubs in publications_dict.items():
    print(f"\n--- {sp} ---")
    if pubs:
        for pub in pubs:
            print(f"{pub['title']}\n  Authors: {pub['authors']}\n  Journal: {pub['journal']} ({pub['year']})\n  PMID: {pub['pmid']} | URL: https://pubmed.ncbi.nlm.nih.gov/{pub['pmid']}/")
    else:
        print("No publications.")

# 6. Phylogenetic tree
seq_dict = {hit['accession']: hit['hit_seq'] for hit in hits}
if len(seq_dict) >= 2:
    alignment, tree = run_clustalo(seq_dict)
    if tree:
        print("\nPhylogenetic Tree (Newick):\n", tree)
    if alignment:
        print("\nAlignment snippet:\n", alignment[:500])
else:
    print("Need at least two sequences for phylogeny.")

# 7. Structure analysis with advanced ColabFold options
os.makedirs('./structures', exist_ok=True)
os.makedirs('./cf_predictions', exist_ok=True)
for hit in hits:
    sp = hit['species']
    acc = hit['accession']
    seq = hit['hit_seq']
    print(f"\n▶ {sp} ({acc})")
    uniprot = get_uniprot_from_accession(acc)
    if not uniprot:
        print("  No UniProt – running ColabFold")
        pdb = run_colabfold(seq, sp.replace(' ', '_'),
                            model_type=model_type,
                            num_recycles=num_recycles,
                            recycle_early_stop_tolerance=recycle_early_stop_tolerance,
                            use_amber=use_amber,
                            use_templates=use_templates,
                            msa_mode=msa_mode,
                            pair_mode=pair_mode,
                            save_all=save_all)
        if pdb:
            display_cif(pdb, f"ColabFold: {sp}")
            plot_ramachandran(pdb, 'pdb', f"{sp} (ColabFold)")
        continue
    print(f"  UniProt: {uniprot}")
    af = check_alphafold_db(uniprot)
    if af:
        print("  ✓ AlphaFold structure found")
        display_cif(af['cif_path'], f"AlphaFold: {sp}")
        plot_plddt(af['cif_path'], uniprot)
        df = create_plddt_table(af['cif_path'], uniprot)
        display(df.head(20))
        if af.get('pae_path'):
            plot_pae(af['pae_path'], uniprot)
        plot_ramachandran(af['cif_path'], 'cif', f"{sp} ({uniprot})")
    else:
        print("  → No AlphaFold – running ColabFold")
        pdb = run_colabfold(seq, sp.replace(' ', '_'),
                            model_type=model_type,
                            num_recycles=num_recycles,
                            recycle_early_stop_tolerance=recycle_early_stop_tolerance,
                            use_amber=use_amber,
                            use_templates=use_templates,
                            msa_mode=msa_mode,
                            pair_mode=pair_mode,
                            save_all=save_all)
        if pdb:
            display_cif(pdb, f"ColabFold: {sp}")
            plot_ramachandran(pdb, 'pdb', f"{sp} (ColabFold)")

print("\n All analyses completed.")

In [ ]:
#@title Download All Results
from google.colab import files as colab_files
import zipfile
import base64

def create_download_zip():
    zip_name = "Seq2Bio_results.zip"
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        # 1. BLAST hits
        if 'hits' in globals() and hits:
            df_blast = pd.DataFrame([
                {"Species": h["species"], "Accession": h["accession"], "Identity": f"{h['identity']:.1%}"}
                for h in hits
            ])
            df_blast.to_csv("blast_results.csv", index=False)
            zipf.write("blast_results.csv")
            os.remove("blast_results.csv")

        # 2. Taxonomy
        if 'taxonomy_dict' in globals() and taxonomy_dict:
            df_tax = pd.DataFrame(taxonomy_dict).T
            df_tax.to_csv("taxonomy.csv")
            zipf.write("taxonomy.csv")
            os.remove("taxonomy.csv")

        # 3. Images (URLs + common names)
        if 'species_images' in globals() and species_images:
            img_rows = []
            for sp, url in species_images.items():
                img_rows.append({
                    "Species": sp,
                    "Image_URL": url if url else "None",
                    "Common_Name": species_common.get(sp, "N/A")
                })
            df_img = pd.DataFrame(img_rows)
            df_img.to_csv("images.csv", index=False)
            zipf.write("images.csv")
            os.remove("images.csv")

        # 4. Occurrences
        if 'occurrences_dict' in globals() and occurrences_dict:
            occ_rows = []
            for sp, occ_list in occurrences_dict.items():
                for occ in occ_list:
                    occ_rows.append({
                        "Species": sp,
                        "Latitude": occ['lat'],
                        "Longitude": occ['lon'],
                        "Location": occ['location']
                    })
            df_occ = pd.DataFrame(occ_rows)
            df_occ.to_csv("occurrences.csv", index=False)
            zipf.write("occurrences.csv")
            os.remove("occurrences.csv")

        # 5. Publications
        if 'publications_dict' in globals() and publications_dict:
            pub_rows = []
            for sp, pubs in publications_dict.items():
                for pub in pubs:
                    pub_rows.append({
                        "Species": sp,
                        "PMID": pub['pmid'],
                        "Title": pub['title'],
                        "Authors": pub['authors'],
                        "Journal": pub['journal'],
                        "Year": pub['year']
                    })
            df_pub = pd.DataFrame(pub_rows)
            df_pub.to_csv("publications.csv", index=False)
            zipf.write("publications.csv")
            os.remove("publications.csv")

        # 6. Phylogenetic tree
        if 'tree' in globals() and tree:
            with open("phylogenetic_tree.nwk", "w") as f:
                f.write(tree)
            zipf.write("phylogenetic_tree.nwk")
            os.remove("phylogenetic_tree.nwk")

        # 7. Structure files
        struct_dirs = ['./structures', './cf_predictions']
        for d in struct_dirs:
            if os.path.exists(d):
                for root, dirs, files in os.walk(d):
                    for file in files:
                        file_path = os.path.join(root, file)
                        arcname = os.path.relpath(file_path, start='.')
                        zipf.write(file_path, arcname)

        # 8. README
        readme = """Seq2Bio Results Archive
========================
This archive contains all results generated by the Seq2Bio pipeline.

Files included:
- blast_results.csv      : BLAST hits with species, accession, identity.
- taxonomy.csv            : Taxonomic lineage for each species.
- images.csv              : Image URLs and common names.
- occurrences.csv         : Geographic occurrence coordinates and locations.
- publications.csv        : PubMed articles related to each species.
- phylogenetic_tree.nwk   : Newick tree file.
- structures/             : Folder with downloaded AlphaFold structures or ColabFold predictions.
- cf_predictions/         : Additional ColabFold predictions (if any).

For questions, contact the author.
"""
        with open("README.txt", "w") as f:
            f.write(readme)
        zipf.write("README.txt")
        os.remove("README.txt")

    time.sleep(1)
    if os.path.exists(zip_name) and os.path.getsize(zip_name) > 0:
        size_mb = os.path.getsize(zip_name) / (1024 * 1024)
        print(f" ZIP file size: {size_mb:.2f} MB")
        colab_files.download(zip_name)
        with open(zip_name, 'rb') as f:
            data = f.read()
        b64 = base64.b64encode(data).decode()
        href = f'<a href="data:application/zip;base64,{b64}" download="{zip_name}" style="font-size:16px; background-color:#4CAF50; color:white; padding:10px 20px; text-decoration:none; border-radius:5px;">📥 Click here to download {zip_name} (fallback)</a>'
        display(HTML(href))
    else:
        print("ZIP file empty or not found.")

create_download_zip()

In [ ]:
#@title Email Notification
if use_email_notification:
    from google.colab import userdata
    import smtplib
    from email.mime.multipart import MIMEMultipart
    from email.mime.text import MIMEText
    from email.mime.application import MIMEApplication

    def send_completion_email(success=True, error_msg=None, zip_path="Seq2Bio_results.zip"):
        try:
            sender_email = userdata.get('EMAIL_ADDRESS')
            sender_password = userdata.get('EMAIL_PASSWORD')
            recipient_email = Entrez.email
            if not sender_email or not sender_password:
                print("⚠️  Email credentials missing. Skipping.")
                return
            if not recipient_email:
                print("⚠️  Recipient email not set. Skipping.")
                return

            msg = MIMEMultipart()
            msg['Subject'] = 'Seq2Bio Analysis Complete'
            msg['From'] = sender_email
            msg['To'] = recipient_email

            body = ""
            if success:
                body = "Your Seq2Bio analysis finished successfully!\n\n"
                if os.path.exists(zip_path):
                    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
                    if size_mb < 20:
                        body += f"Results attached (ZIP, {size_mb:.2f} MB).\n\n"
                        with open(zip_path, 'rb') as f:
                            attachment = MIMEApplication(f.read(), Name=os.path.basename(zip_path))
                        attachment['Content-Disposition'] = f'attachment; filename="{os.path.basename(zip_path)}"'
                        msg.attach(attachment)
                    else:
                        body += f"Results ZIP too large ({size_mb:.2f} MB). Please download manually.\n"
                else:
                    body += "Note: Results ZIP not found.\n"
            else:
                body = f"Seq2Bio encountered an error:\n\n{error_msg}"

            msg.attach(MIMEText(body, 'plain'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
                smtp.login(sender_email, sender_password)
                smtp.send_message(msg)
            print("📧 Email sent.")
        except Exception as e:
            print(f"❌ Email failed: {e}")

    if 'hits' in globals() and hits:
        send_completion_email(success=True)
    else:
        send_completion_email(success=False, error_msg="No BLAST hits found.")
else:
    print("Email notification disabled.")